In [1]:
import matplotlib.pyplot as plt
import numpy as np
import utils
import pandas as pd

audio_paths = [
    "../music-clips/classical_1_60s.wav",
    # "../music-clips/classical_1_30s.wav",
    # "../music-clips/classical_1_10s.wav",
    "../music-clips/electronic_5_60s.wav",
    "../music-clips/cicadas_57s.wav",
    # "../music-clips/electronic_10.wav",
    "../music-clips/guitar_melody_131s.wav",

    "../music-clips/beep_30s.wav",
    "../music-clips/crickets_6s.wav",
    "../music-clips/metronome_60s.wav",
    # "../music-clips/sa01.wav",
    # "../music-clips/sa02.wav",
    ]

model, processor = utils.load_music_gen_model()
wfs = utils.load_process_bulk_audio(audio_paths, sr=32000)
inputs_dict = utils.process_bulk_music_gen(wfs, processor=processor)

attentions_dict = utils.extract_attentions(model, inputs_dict)

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

In [2]:
def extract_and_rank_all_384_heads(attentions_dict, normalize=True):
    """
    Averages MAD across all clips in attentions_dict and ranks all 384 coordinates.
    
    Returns:
        df_ranked (pd.DataFrame): DataFrame of 384 coordinates sorted by MAD.
        avg_mad_matrix (np.ndarray): 24x16 matrix of averaged MAD values.
    """
    all_sample_mads = []

    # Compute MAD matrix for each audio sample
    for path, sample_attentions in attentions_dict.items():
        sample_mad_matrix = utils.compute_mad_by_layer(
            sample_attentions, sample_attentions[0].shape[-1]
        )
        all_sample_mads.append(sample_mad_matrix)

    # Average across all audio clips
    avg_mad_matrix = np.mean(np.array(all_sample_mads), axis=0)
    num_layers, num_heads = avg_mad_matrix.shape

    # Flatten into 384 individual coordinates
    records = []
    for l in range(num_layers):
        for h in range(num_heads):
            records.append({
                "layer": l,
                "head": h,
                "coordinate": f"L{l}H{h}",
                "mad": avg_mad_matrix[l, h],
                "mad_percent": f"{avg_mad_matrix[l, h] * 100:.2f}%" if normalize else f"{avg_mad_matrix[l, h]:.1f} tokens"
            })

    df_ranked = pd.DataFrame(records).sort_values(by="mad", ascending=False).reset_index(drop=True)
    df_ranked.index += 1  # 1-based rank
    df_ranked.index.name = "Rank"

    return df_ranked, avg_mad_matrix

In [3]:
df_ranked, avg_mad_matrix = extract_and_rank_all_384_heads(attentions_dict, normalize=True)

# 2. Display the top 15 Global Integration Heads
print("=== TOP 15 GLOBAL INTEGRATION HEADS ===")
print(df_ranked.head(15)[["layer", "head", "coordinate", "mad_percent"]])

=== TOP 15 GLOBAL INTEGRATION HEADS ===
      layer  head coordinate mad_percent
Rank                                    
1         6     3       L6H3      38.67%
2         7     0       L7H0      37.91%
3         7     8       L7H8      37.84%
4         6     9       L6H9      37.10%
5         7     3       L7H3      36.88%
6         8     5       L8H5      36.79%
7         9    13      L9H13      36.79%
8         3     1       L3H1      36.54%
9         8    15      L8H15      36.16%
10        9     7       L9H7      36.06%
11        7     4       L7H4      35.80%
12        4     5       L4H5      35.72%
13        8     8       L8H8      35.29%
14        4    14      L4H14      35.24%
15       12     5      L12H5      35.04%
